# Статистический анализ результатов поиска (Wilcoxon Signed-Rank Test)

Сравниваем распределение метрики Reciprocal Rank (RR) для каждого вопроса из валидационного датасета между базовой (например, BM25 или Dense) и оптимизированной стратегией поиска (Hybrid + Rerank). 

Поскольку распределение метрики RR дискретное (может принимать значения 1, 0.5, 0.33, 0.25, ..., 0) и скошенное, используется **непараметрический критерий знаковых рангов Уилкоксона** (Wilcoxon signed-rank test).

In [1]:
import json
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

def load_metrics(json_path: str):
    """Загружает метрики из JSON-файла и гарантирует консистентную сортировку вопросов."""
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")
        
    data = json.loads(path.read_text(encoding='utf-8'))
    q_data = sorted(data['per_question'], key=lambda x: x['id'])
    return data.get('tag', path.stem), [q['rr'] for q in q_data]

In [ ]:
# ПУТИ К JSON-ФАЙЛАМ ОТ СРИПТА evaluate_retrieval.py 
# path_baseline = '../experiments/ab_e5_c512_naive/metrics_ab-e5-naive-c512_20261028.json'
# path_best = '../experiments/user_bge_m3_c2048/metrics_user_bge_m3-c2048-hybrid-rerank_20261028.json'

path_baseline = 'ЗДЕСЬ_ПУТЬ_К_БАЗОВОЙ_МОДЕЛИ.json'
path_best = 'ЗДЕСЬ_ПУТЬ_К_ОПТИМИЗИРОВАННОЙ_МОДЕЛИ.json'

# Заглушка для демонстрации
# -------------------------------------------------------------------------
import tempfile
d_base = {"tag": "baseline", "per_question": [{"id": f"q{i}", "rr": 0.5} for i in range(200)]}
d_best = {"tag": "best", "per_question": [{"id": f"q{i}", "rr": 1.0 if i % 2 == 0 else 0.5} for i in range(200)]}
with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.json') as f1, tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.json') as f2:
    json.dump(d_base, f1)
    json.dump(d_best, f2)
    path_baseline, path_best = f1.name, f2.name
# -------------------------------------------------------------------------

tag_base, rr_base = load_metrics(path_baseline)
tag_best, rr_best = load_metrics(path_best)

print(f"Сравниваем:\n[A] {tag_base} (Baseline)\n[B] {tag_best} (Best)")

Сравниваем:
[A] baseline (Baseline)
[B] best (Best)


In [3]:
# 1. Считаем макро-усреднение (MRR)
mrr_base = sum(rr_base) / len(rr_base)
mrr_best = sum(rr_best) / len(rr_best)

print(f"MRR [A]: {mrr_base:.4f}")
print(f"MRR [B]: {mrr_best:.4f}")
print(f"Абсолютный прирост: +{(mrr_best - mrr_base):.4f}\n")

# 2. Выполняем тест Уилкоксона
# alternative='less' означает проверку гипотезы: "Правда ли, что распределение A меньше распределения B?"
stat, p_value = wilcoxon(rr_base, rr_best, alternative='less')

print(f"Wilcoxon statistic (W): {stat}")
print(f"p-value: {p_value:.6e}")

if p_value < 0.05:
    print("\nВЫВОД: Улучшение статистически значимо (p < 0.05). Прирост обусловлен архитектурой, а не случайностью.")
else:
    print("\nВЫВОД: Разница НЕ является статистически значимой. Улучшение находится в пределах погрешности/шума.")

MRR [A]: 0.5000
MRR [B]: 0.7500
Абсолютный прирост: +0.2500

Wilcoxon statistic (W): 0.0
p-value: 7.619853e-24

ВЫВОД: Улучшение статистически значимо (p < 0.05). Прирост обусловлен архитектурой, а не случайностью.
